In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import logging
from glob import glob
import random
import glob
from multiprocessing import Pool

In [ ]:
imu_sensor_locations = ['Pelvis', 'L5', 'L3', 'T12', 'T8', 'Neck', 'Head', 'RightShoulder', 'RightUpperArm', 'RightForeArm', 'RightHand', 'LeftShoulder', 'LeftUpperArm', 'LeftForeArm', 'LeftHand', 'RightUpperLeg', 'RightLowerLeg',
                    'RightFoot', 'RightToe', 'LeftUpperLeg', 'LeftLowerLeg', 'LeftFoot', 'LeftToe']

luo_sensor_locations = ['Pelvis', 'RightForeArm', 'RightUpperLeg', 'RightLowerLeg', 'LeftUpperLeg', 'LeftLowerLeg']

lower_body_sensor_locations = ['Pelvis', 'RightUpperLeg', 'RightLowerLeg', 'LeftUpperLeg', 'LeftLowerLeg', 'RightFoot', 'LeftFoot']

insole_sensor_locations = ['Arch', 'Hallux', 'Heel_L', 'Heel_R', 'Met1', 'Met3', 'Met5', 'Toes']

participant_num = [1, 2, 3, 4, 5, 7, 8, 10, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25]

##  insole df and imu df import & random segmentation selection

In [ ]:
# import base insole and base imu
insole_df = pd.read_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/base_df_insole_df.csv')

In [ ]:
insole_df = insole_df[['time', 'participant_id', 'task', 'sensor_location',
                       'Left_norm', 'Left_raw', 'Right_norm', 'Right_raw',
                       'Left_norm_cumulative', 'Right_norm_cumulative']]

In [ ]:
insole_df['gait_cycle_id'] = insole_df['participant_id'].astype(str) + '_' + insole_df['task'].astype(str) + '_' + insole_df['time'].astype(str)

In [ ]:
imu_df = pd.read_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/base_df_imu_df.csv')

In [ ]:
imu_df = imu_df[['time', 'participant_id', 'task', 'walk_mode', 'sensor_location',
                 'acceleration_x', 'acceleration_y', 'acceleration_z',
                 'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z']]

In [ ]:
insole_df['time'].equals(imu_df['time'])

True

In [ ]:
insole_df = insole_df[['time', 'participant_id', 'task', 'Left_norm', 'Right_norm', 'Left_norm_cumulative', 'Right_norm_cumulative', 'gait_cycle_id']]

In [ ]:
combo_df = pd.merge(imu_df, insole_df, on=['participant_id', 'task', 'time'], suffixes=('_imu', '_insole'), how='outer')

In [ ]:
del insole_df
del imu_df

In [ ]:
combo_df.columns

Index(['time', 'participant_id', 'task', 'walk_mode', 'sensor_location',
       'acceleration_x', 'acceleration_y', 'acceleration_z',
       'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z',
       'Left_norm', 'Right_norm', 'Left_norm_cumulative',
       'Right_norm_cumulative', 'gait_cycle_id'],
      dtype='object')

In [ ]:
combo_no_dupes = combo_df.drop_duplicates(subset=['time', 'participant_id', 'task'])

In [ ]:
len(combo_no_dupes)

2026204

In [ ]:
del combo_df

In [ ]:
def sliding_window(data, window_size, step_size):
    num_segments = (len(data) - window_size) // step_size + 1
    indices = np.arange(0, num_segments * step_size, step_size)
    return [data[i:i + window_size] for i in indices]

segment_size = 85
overlap = 0.75
step_size = int(segment_size * (1 - overlap))
print(f"Step size: {step_size}")
participants = combo_no_dupes['participant_id'].unique()
tasks = combo_no_dupes['task'].unique()

for p in participants:
    print(f"Processing participant: {p}")
    participant_df = combo_no_dupes[combo_no_dupes['participant_id'] == p]
    participant_segments = []

    for t in tasks:
        print(f"Processing task: {t}")
        task_df = participant_df[participant_df['task'] == t]

        unique_times = task_df['time'].unique()
        if len(unique_times) < segment_size:
            continue  # Skip if not enough data for a segment

        count = 0
        for segment in sliding_window(unique_times, segment_size, step_size):
            count += 1
            segment_mask = task_df['time'].isin(segment)
            segment_df = task_df.loc[segment_mask]
            segment_df['gait_cycle_id'] = f'{p}_{t}_{count}'
            segment_df['segment_count'] = count

            # Check for uniform surface type
            if segment_df['walk_mode'].nunique() == 1:
                min_time, max_time = segment_df['time'].min(), segment_df['time'].max()
                participant_segments.append(segment_df)


    # concat
    participant_segments_df = pd.concat(participant_segments, ignore_index=True)
    participant_segments_df.to_csv(f'/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/alt_seg/ppant_{p}_segments.csv', index=False)

print("Segmentation complete.")


Output hidden; open in https://colab.research.google.com to view.

## reupload of randomly selected segments (to save RAM)

In [18]:
segmented_dfs = []

# navigate to folder and upload csvs in folder
os.chdir('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/alt_seg')

for file in glob.glob("*.csv"):
    # import to df
    df = pd.read_csv(file)
    segmented_dfs.append(df)



In [ ]:
# segmented_dfs[1]

In [19]:
for df in range(len(segmented_dfs)):
    # keep only specific columns
    segmented_dfs[df] = segmented_dfs[df][['time', 'participant_id', 'task', 'walk_mode', 'gait_cycle_id', 'segment_count']]


In [20]:
segmented_dfs[0]

,time,participant_id,task,walk_mode,gait_cycle_id,segment_count
0,0,1,A,walk,1_A_1,1
1,16,1,A,walk,1_A_1,1
2,33,1,A,walk,1_A_1,1
3,50,1,A,walk,1_A_1,1
4,66,1,A,walk,1_A_1,1
...,...,...,...,...,...,...
371190,404533,1,C,walk,1_C_1153,1153
371191,404550,1,C,walk,1_C_1153,1153
371192,404566,1,C,walk,1_C_1153,1153
371193,404583,1,C,walk,1_C_1153,1153


In [21]:
alt_seg_df = pd.concat(segmented_dfs, ignore_index=True)

## insole + segmentations

In [22]:
# import base insole and base imu
insole_df = pd.read_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/base_df_insole_df.csv')

In [23]:
insole_df = insole_df[['time', 'participant_id', 'task', 'sensor_location', 'walk_mode', 'Left_norm', 'Right_norm', 'Left_norm_cumulative', 'Right_norm_cumulative']]

In [24]:
# merge alt_seg left onto insole_df
combo_df = pd.merge(insole_df, alt_seg_df, on=['participant_id', 'task', 'time'], suffixes=('_insole', '_alt_seg'), how='left')

In [25]:
# combo_df.head()

In [26]:
# combo_df['gait_cycle_id'].value_counts()

In [27]:
# export to csv
combo_df.to_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/alt_seg_insole_combo_df.csv', index=False)

In [28]:
del combo_df

## imu + segmentations

In [ ]:
imu_df = pd.read_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/base_df_imu_df.csv')

In [ ]:
imu_df = imu_df[['time', 'participant_id', 'task', 'walk_mode', 'sensor_location',
                 'acceleration_x', 'acceleration_y', 'acceleration_z',
                 'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z']]

In [ ]:
# merge alt_seg left onto insole_df
combo_df = pd.merge(imu_df, alt_seg_df, on=['participant_id', 'task', 'time'], suffixes=('_insole', '_alt_seg'), how='left')

In [ ]:
# combo_df['sensor_location'].value_counts()

In [ ]:
# combo_df['gait_cycle_id'].value_counts()

In [ ]:
# export to csv
combo_df.to_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/alt_seg_imu_combo_df.csv', index=False)

In [ ]:
del combo_df

## insole reupload segmented before interpolation

In [29]:
# label encoding the task column
from sklearn.preprocessing import LabelEncoder

In [30]:
# import base insole
insole_df = pd.read_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/alt_seg_insole_combo_df.csv')

In [31]:
insole_df.head(250)

,time,participant_id,task,sensor_location,walk_mode_insole,Left_norm,Right_norm,Left_norm_cumulative,Right_norm_cumulative,walk_mode_alt_seg,gait_cycle_id,segment_count
0,0,1,A,Arch,walk,0.001299,0.160105,0.838253,3.509033,walk,1_A_1,1.0
1,0,1,A,Hallux,walk,0.000223,0.968475,0.838253,3.509033,walk,1_A_1,1.0
2,0,1,A,Heel_L,walk,0.457522,0.000275,0.838253,3.509033,walk,1_A_1,1.0
3,0,1,A,Heel_R,walk,0.372672,0.002728,0.838253,3.509033,walk,1_A_1,1.0
4,0,1,A,Met1,walk,0.001484,0.987995,0.838253,3.509033,walk,1_A_1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
245,416,1,A,Met5,walk,0.674458,0.002136,2.494302,0.227988,walk,1_A_2,2.0
246,416,1,A,Toes,walk,0.060901,0.000050,2.494302,0.227988,walk,1_A_1,1.0
247,416,1,A,Toes,walk,0.060901,0.000050,2.494302,0.227988,walk,1_A_2,2.0
248,433,1,A,Arch,walk,1.000000,0.060116,2.551036,0.227942,walk,1_A_1,1.0


In [32]:
# make letter column numeric
le = LabelEncoder()
insole_df['task'] = le.fit_transform(insole_df['task'])
insole_df['walk_mode'] = le.fit_transform(insole_df['walk_mode_insole'])

In [33]:
insole_df[['walk_mode', 'walk_mode_insole']].value_counts()

,,count
walk_mode,walk_mode_insole,
4,walk,33407336
0,slope_down,11067672
1,slope_up,8240040
3,stairs_up,4760648
2,stairs_down,2710640


In [34]:
insole_df.sort_values(by=['participant_id', 'task', 'segment_count', 'time'])

,time,participant_id,task,sensor_location,walk_mode_insole,Left_norm,Right_norm,Left_norm_cumulative,Right_norm_cumulative,walk_mode_alt_seg,gait_cycle_id,segment_count,walk_mode
0,0,1,0,Arch,walk,0.001299,0.160105,0.838253,3.509033,walk,1_A_1,1.0,4
1,0,1,0,Hallux,walk,0.000223,0.968475,0.838253,3.509033,walk,1_A_1,1.0,4
2,0,1,0,Heel_L,walk,0.457522,0.000275,0.838253,3.509033,walk,1_A_1,1.0,4
3,0,1,0,Heel_R,walk,0.372672,0.002728,0.838253,3.509033,walk,1_A_1,1.0,4
4,0,1,0,Met1,walk,0.001484,0.987995,0.838253,3.509033,walk,1_A_1,1.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
60186331,399517,25,2,Heel_R,walk,0.000595,0.784188,0.688663,2.909417,NaN,NaN,NaN,4
60186332,399517,25,2,Met1,walk,0.285393,0.004241,0.688663,2.909417,NaN,NaN,NaN,4
60186333,399517,25,2,Met3,walk,0.056447,0.049854,0.688663,2.909417,NaN,NaN,NaN,4
60186334,399517,25,2,Met5,walk,0.004141,0.321895,0.688663,2.909417,NaN,NaN,NaN,4


## imu reupload segmented before interpolation

In [ ]:
# import base imu
imu_df = pd.read_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/alt_seg_imu_combo_df.csv')

In [ ]:
imu_df.head()

In [ ]:
# label encoding the task column
from sklearn.preprocessing import LabelEncoder

# make letter column numeric
le = LabelEncoder()
imu_df['task'] = le.fit_transform(imu_df['task'])
imu_df['walk_mode'] = le.fit_transform(imu_df['walk_mode_insole'])

In [ ]:
imu_df.sort_values(by=['participant_id', 'task', 'segment_count', 'time'])

## interpolation

In [35]:
def interpolate_to_fixed_length(group, target_length=80):
    interpolated_group = []

    # Iterate over each sensor location within the group
    for sensor_location, data in group.groupby('sensor_location'):
        # Select only numeric columns for interpolation
        numeric_data = data.select_dtypes(include=[np.number])
        numeric_data = numeric_data.reset_index(drop=True)
        original_length = len(numeric_data)

        if original_length < 2:
            # Skip groups with insufficient data for interpolation
            print(f"Skipping sensor_location '{sensor_location}' with insufficient data.")
            continue

        # Create new evenly spaced indices
        new_indices = np.linspace(0, original_length - 1, target_length)

        # Interpolate the numeric data to the new indices
        interpolated_data = pd.DataFrame(
            {col: np.interp(new_indices, np.arange(original_length), numeric_data[col])
             for col in numeric_data.columns},
            index=new_indices
        )

        # Add back the sensor location information
        interpolated_data['sensor_location'] = sensor_location
        interpolated_data['walk_mode'] = group['walk_mode'].iloc[0]

        # Append to the results list
        interpolated_group.append(interpolated_data)

    # Combine all sensor locations back into a single DataFrame
    return pd.concat(interpolated_group, axis=0) if interpolated_group else pd.DataFrame()



def process_gait_cycle(group):
    return interpolate_to_fixed_length(group, target_length=80)


In [38]:
# Define the interpolation function
def interpolate_group(df, target_length=80):
    # Use your custom interpolation logic
    # Assuming 'time' is the x-axis and other columns are y-values
    df = df.sort_values(by='time')  # Ensure sorting
    interpolated = interpolate_to_fixed_length(df, target_length=target_length)
    return interpolated

# Group the DataFrame by 'participant_id' and 'gait_cycle_id'
grouped = insole_df.groupby(['participant_id', 'gait_cycle_id'])

# Apply the interpolation to each group
interpolated_results = grouped.apply(lambda group: interpolate_group(group))

# Reset index if needed
interpolated_results = interpolated_results.reset_index(drop=True)


<ipython-input-38-37fadfefb6c7>:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  interpolated_results = grouped.apply(lambda group: interpolate_group(group))


In [37]:
# # Group the DataFrame by 'participant_id' and 'gait_cycle_id'
# grouped = insole_df.groupby(['participant_id', 'gait_cycle_id'])

# # Apply the interpolation to each group
# interpolated_results = grouped.apply(lambda group: interpolate_group(group))

# # Reset index if needed
# interpolated_results = interpolated_results.reset_index(drop=True)


In [39]:
participant_1 = interpolated_results[interpolated_results['participant_id'] == 1]

In [40]:
participant_1

,time,participant_id,task,Left_norm,Right_norm,Left_norm_cumulative,Right_norm_cumulative,segment_count,walk_mode,sensor_location
0,0.000000,1.0,0.0,0.001299,0.160105,0.838253,3.509033,1.0,4,Arch
1,17.075949,1.0,0.0,0.008923,0.151230,1.193033,3.197767,1.0,4,Arch
2,35.151899,1.0,0.0,0.169538,0.158577,1.489439,2.786481,1.0,4,Arch
3,53.037975,1.0,0.0,0.387587,0.152760,1.554394,2.201408,1.0,4,Arch
4,70.303797,1.0,0.0,0.506269,0.126174,1.662415,1.567283,1.0,4,Arch
...,...,...,...,...,...,...,...,...,...,...
2794875,350628.696203,1.0,2.0,0.001060,0.000781,0.005812,2.120614,999.0,4,Toes
2794876,350646.772152,1.0,2.0,0.000407,0.001702,0.010379,2.074832,999.0,4,Toes
2794877,350663.974684,1.0,2.0,0.001647,0.001521,0.012682,1.972878,999.0,4,Toes
2794878,350681.924051,1.0,2.0,0.001503,0.002678,0.011087,1.888742,999.0,4,Toes


In [41]:
participant_1[['participant_id', 'task', 'segment_count']].value_counts()

participant_id  task  segment_count
1.0             0.0   1.0              640
                1.0   1198.0           640
                      1208.0           640
                      1207.0           640
                      1206.0           640
                                      ... 
                0.0   1614.0           640
                      1615.0           640
                      1616.0           640
                      1617.0           640
                2.0   1153.0           640
Name: count, Length: 4367, dtype: int64

In [ ]:
# interpolated_results.to_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/interpolated_alt_seg_imu_sensor_df.csv', index=False)

In [42]:
interpolated_results.to_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/interpolated_alt_seg_insole_sensor_df.csv', index=False)